
# Physics-Informed Residual MLP Baseline for Fisher-Regularized Variance Dynamics

This notebook computes the comparison table for:

- Direct MLP
- Residual MLP
- Physics-informed Residual MLP with \(\lambda\in\{0.01,0.1,1.0\}\)

The reduced Fisher-regularized variance dynamics are

\[
\dot{u}=2(1-u)+\frac{\varepsilon}{u}.
\]

The physics-informed residual MLP adds an ODE residual penalty to the trajectory loss:

\[
\mathcal{L}=\mathcal{L}_{\mathrm{traj}}+\lambda\mathcal{L}_{\mathrm{phys}}.
\]

The notebook reports \(E_{\mathrm{traj}}\), \(E_{\mathrm{cross}}\), \(E_{\mathrm{over}}\), and \(E_{\mathrm{eq}}\) as mean \(\pm\) 95% confidence-interval half-width over \(S=10\) random seeds.


In [ ]:

import math
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

try:
    from scipy.stats import t as student_t
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


PyTorch version: 2.11.0+cpu
CUDA available: False


## Configuration

In [ ]:

CONFIG = {
    'n_pairs': 300,
    'n_time': 80,
    't_max': 5.0,
    'u0_min': 0.2,
    'u0_max': 2.5,
    'eps_min': 0.01,
    'eps_max': 0.5,
    'n_seeds': 10,
    'data_seed': 123,
    'split_seed': 999,
    'base_seed': 2026,
    'epochs': 1500,
    'batch_size': 512,
    'lr': 2e-3,
    'patience': 150,
    'physics_lambdas': [0.01, 0.1, 1.0],
    # Set True for quick debugging.
    'quick_test': False,
}

if CONFIG['quick_test']:
    CONFIG.update({
        'n_pairs': 80,
        'n_time': 50,
        'n_seeds': 2,
        'epochs': 200,
        'patience': 40,
    })

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE


'cpu'

## Fisher-Regularized Dynamics

In [ ]:

def u_star(eps):
    eps = np.asarray(eps)
    return (1.0 + np.sqrt(1.0 + 2.0 * eps)) / 2.0


def u_ou_np(t, u0):
    return 1.0 + (u0 - 1.0) * np.exp(-2.0 * t)


def u_ou_torch(t, u0):
    return 1.0 + (u0 - 1.0) * torch.exp(-2.0 * t)


def rhs(u, eps):
    return 2.0 * (1.0 - u) + eps / np.maximum(u, 1e-12)


def integrate_rk4(u0, eps, t_grid):
    u = np.zeros_like(t_grid, dtype=np.float64)
    u[0] = float(u0)
    for k in range(1, len(t_grid)):
        dt = t_grid[k] - t_grid[k - 1]
        uk = u[k - 1]
        k1 = rhs(uk, eps)
        k2 = rhs(uk + 0.5 * dt * k1, eps)
        k3 = rhs(uk + 0.5 * dt * k2, eps)
        k4 = rhs(uk + dt * k3, eps)
        u[k] = uk + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)
        u[k] = max(u[k], 1e-12)
    return u


## Dataset Generation

In [ ]:

def make_dataset(n_pairs, n_time, u0_range=(0.2, 2.5), eps_range=(0.01, 0.5), t_max=5.0, seed=123):
    rng = np.random.default_rng(seed)
    u0s = rng.uniform(u0_range[0], u0_range[1], size=n_pairs)
    epss = rng.uniform(eps_range[0], eps_range[1], size=n_pairs)
    t_grid = np.linspace(0.0, t_max, n_time)
    rows = []
    for pair_id, (u0_i, eps_i) in enumerate(zip(u0s, epss)):
        u_clean = integrate_rk4(u0_i, eps_i, t_grid)
        for t_i, u_i in zip(t_grid, u_clean):
            rows.append({'pair_id': pair_id, 'u0': u0_i, 'eps': eps_i, 't': t_i, 'u_clean': u_i})
    return pd.DataFrame(rows)


def split_pair_ids(n_pairs, train_frac=0.70, val_frac=0.15, seed=999):
    rng = np.random.default_rng(seed)
    ids = np.arange(n_pairs)
    rng.shuffle(ids)
    n_train = int(train_frac * n_pairs)
    n_val = int(val_frac * n_pairs)
    return set(ids[:n_train]), set(ids[n_train:n_train+n_val]), set(ids[n_train+n_val:])


df = make_dataset(
    n_pairs=CONFIG['n_pairs'],
    n_time=CONFIG['n_time'],
    u0_range=(CONFIG['u0_min'], CONFIG['u0_max']),
    eps_range=(CONFIG['eps_min'], CONFIG['eps_max']),
    t_max=CONFIG['t_max'],
    seed=CONFIG['data_seed'],
)

train_ids, val_ids, test_ids = split_pair_ids(CONFIG['n_pairs'], seed=CONFIG['split_seed'])
train_df = df[df['pair_id'].isin(train_ids)].copy()
val_df = df[df['pair_id'].isin(val_ids)].copy()
test_df = df[df['pair_id'].isin(test_ids)].copy()

print(df.head())
print('Train/Val/Test shapes:', train_df.shape, val_df.shape, test_df.shape)


   pair_id        u0       eps         t   u_clean
0        0  1.769409  0.145374  0.000000  1.769409
1        0  1.769409  0.145374  0.063291  1.682941
2        0  1.769409  0.145374  0.126582  1.607001
3        0  1.769409  0.145374  0.189873  1.540327
4        0  1.769409  0.145374  0.253165  1.481808
Train/Val/Test shapes: (16800, 5) (3600, 5) (3600, 5)


## Model and Scaling Utilities

In [ ]:

class MLP(nn.Module):
    def __init__(self, input_dim=3, hidden=(64, 64), activation='tanh'):
        super().__init__()
        if activation == 'relu':
            act = nn.ReLU
        elif activation == 'gelu':
            act = nn.GELU
        else:
            act = nn.Tanh
        layers = []
        prev = input_dim
        for h in hidden:
            layers.append(nn.Linear(prev, h))
            layers.append(act())
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)


@dataclass
class Scaler:
    mean: np.ndarray
    std: np.ndarray
    def transform(self, x):
        return (x - self.mean) / self.std
    def inverse(self, x):
        return x * self.std + self.mean


def fit_scaler(x):
    mean = x.mean(axis=0, keepdims=True)
    std = x.std(axis=0, keepdims=True)
    std = np.where(std < 1e-12, 1.0, std)
    return Scaler(mean=mean, std=std)


def scaler_to_torch(scaler, device):
    mean = torch.tensor(scaler.mean, dtype=torch.float32, device=device)
    std = torch.tensor(scaler.std, dtype=torch.float32, device=device)
    return mean, std


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


## Direct and Residual MLP Training

In [ ]:

def train_scaled_model(x_train, y_train, x_val, y_val, seed, epochs=1500, batch_size=512, lr=2e-3, patience=150, device='cpu'):
    set_seed(seed)
    x_scaler = fit_scaler(x_train)
    y_scaler = fit_scaler(y_train)
    x_train_s = x_scaler.transform(x_train).astype(np.float32)
    y_train_s = y_scaler.transform(y_train).astype(np.float32)
    x_val_s = x_scaler.transform(x_val).astype(np.float32)
    y_val_s = y_scaler.transform(y_val).astype(np.float32)
    train_ds = TensorDataset(torch.tensor(x_train_s), torch.tensor(y_train_s))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    model = MLP(input_dim=x_train.shape[1], hidden=(64, 64), activation='tanh').to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    x_val_t = torch.tensor(x_val_s, dtype=torch.float32, device=device)
    y_val_t = torch.tensor(y_val_s, dtype=torch.float32, device=device)
    best_state = None
    best_val = float('inf')
    wait = 0
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(x_val_t), y_val_t).item()
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, x_scaler, y_scaler


def predict_scaled_model(model, x, x_scaler, y_scaler, device='cpu'):
    model.eval()
    x_s = x_scaler.transform(x).astype(np.float32)
    with torch.no_grad():
        pred_s = model(torch.tensor(x_s, dtype=torch.float32, device=device)).cpu().numpy()
    return y_scaler.inverse(pred_s)


## Physics-Informed Residual MLP Training

In [ ]:

def train_physics_informed_residual(train_df, val_df, lam, seed, epochs=1500, batch_size=512, lr=2e-3, patience=150, device='cpu'):
    set_seed(seed)
    x_train = train_df[['u0', 'eps', 't']].to_numpy(dtype=np.float64)
    x_val = val_df[['u0', 'eps', 't']].to_numpy(dtype=np.float64)
    y_train_u = train_df[['u_clean']].to_numpy(dtype=np.float64)
    y_val_u = val_df[['u_clean']].to_numpy(dtype=np.float64)

    train_ou = u_ou_np(train_df['t'].to_numpy(), train_df['u0'].to_numpy())
    train_eps = train_df['eps'].to_numpy()
    y_train_r = ((train_df['u_clean'].to_numpy() - train_ou) / train_eps).reshape(-1, 1)

    x_scaler = fit_scaler(x_train)
    r_scaler = fit_scaler(y_train_r)
    x_mean_t, x_std_t = scaler_to_torch(x_scaler, device)
    r_mean_t, r_std_t = scaler_to_torch(r_scaler, device)

    train_tensor = torch.tensor(x_train.astype(np.float32))
    train_u_tensor = torch.tensor(y_train_u.astype(np.float32))
    train_ds = TensorDataset(train_tensor, train_u_tensor)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model = MLP(input_dim=3, hidden=(64, 64), activation='tanh').to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    x_val_raw = torch.tensor(x_val.astype(np.float32), device=device)
    y_val_u_t = torch.tensor(y_val_u.astype(np.float32), device=device)
    best_state = None
    best_val = float('inf')
    wait = 0

    for epoch in range(epochs):
        model.train()
        for xb_raw_cpu, yb_u_cpu in train_loader:
            xb_raw = xb_raw_cpu.to(device).clone().detach().requires_grad_(True)
            yb_u = yb_u_cpu.to(device)
            u0 = xb_raw[:, 0:1]
            eps = xb_raw[:, 1:2]
            t = xb_raw[:, 2:3]
            xb_norm = (xb_raw - x_mean_t) / x_std_t
            r_scaled = model(xb_norm)
            r_pred = r_scaled * r_std_t + r_mean_t
            u_hat = u_ou_torch(t, u0) + eps * r_pred
            u_hat_safe = torch.clamp(u_hat, min=1e-6)
            loss_traj = torch.mean((u_hat - yb_u) ** 2)
            du_dt = torch.autograd.grad(outputs=u_hat.sum(), inputs=xb_raw, create_graph=True, retain_graph=True)[0][:, 2:3]
            ode_rhs = 2.0 * (1.0 - u_hat_safe) + eps / u_hat_safe
            residual = du_dt - ode_rhs
            loss_phys = torch.mean(residual ** 2)
            loss = loss_traj + lam * loss_phys
            opt.zero_grad()
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            x_val_norm = (x_val_raw - x_mean_t) / x_std_t
            u0_val = x_val_raw[:, 0:1]
            eps_val = x_val_raw[:, 1:2]
            t_val = x_val_raw[:, 2:3]
            r_val_scaled = model(x_val_norm)
            r_val = r_val_scaled * r_std_t + r_mean_t
            u_val_hat = u_ou_torch(t_val, u0_val) + eps_val * r_val
            val_loss = torch.mean((u_val_hat - y_val_u_t) ** 2).item()
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, x_scaler, r_scaler


def predict_physics_informed_residual(model, x, x_scaler, r_scaler, device='cpu'):
    model.eval()
    x_raw = torch.tensor(x.astype(np.float32), device=device)
    x_mean_t, x_std_t = scaler_to_torch(x_scaler, device)
    r_mean_t, r_std_t = scaler_to_torch(r_scaler, device)
    with torch.no_grad():
        x_norm = (x_raw - x_mean_t) / x_std_t
        u0 = x_raw[:, 0:1]
        eps = x_raw[:, 1:2]
        t = x_raw[:, 2:3]
        r_scaled = model(x_norm)
        r_pred = r_scaled * r_std_t + r_mean_t
        u_hat = u_ou_torch(t, u0) + eps * r_pred
    return u_hat.cpu().numpy()


## Evaluation Metrics

In [ ]:

def crossing_time(t, u, level=1.0):
    if u[0] >= level:
        return None
    above = np.where(u >= level)[0]
    if len(above) == 0:
        return None
    k = above[0]
    if k == 0:
        return t[0]
    t0, t1 = t[k - 1], t[k]
    u0_, u1_ = u[k - 1], u[k]
    if abs(u1_ - u0_) < 1e-12:
        return t1
    return t0 + (level - u0_) * (t1 - t0) / (u1_ - u0_)


def compute_metrics(test_pred_df):
    e_traj_values, e_cross_values, e_over_values, e_eq_values = [], [], [], []
    for pair_id, g in test_pred_df.groupby('pair_id'):
        g = g.sort_values('t')
        t = g['t'].to_numpy()
        u_true = g['u_clean'].to_numpy()
        u_pred = g['u_pred'].to_numpy()
        eps = float(g['eps'].iloc[0])
        ueq = float(u_star(np.array([eps]))[0])
        e_traj_values.append(np.mean((u_pred - u_true) ** 2))
        tc_true = crossing_time(t, u_true, level=1.0)
        if tc_true is not None:
            tc_pred = crossing_time(t, u_pred, level=1.0)
            if tc_pred is None:
                tc_pred = t[-1]
            e_cross_values.append(abs(tc_pred - tc_true))
        a_true = max(0.0, float(np.max(u_true) - ueq))
        a_pred = max(0.0, float(np.max(u_pred) - ueq))
        e_over_values.append(abs(a_pred - a_true))
        e_eq_values.append(abs(float(u_pred[-1]) - ueq))
    return {
        'E_traj': float(np.mean(e_traj_values)),
        'E_cross': float(np.mean(e_cross_values)) if len(e_cross_values) else np.nan,
        'E_over': float(np.mean(e_over_values)),
        'E_eq': float(np.mean(e_eq_values)),
    }


## Run Baseline Comparison

In [ ]:

def run_physics_baseline_comparison(config=CONFIG, device=DEVICE):
    all_rows = []
    x_train = train_df[['u0', 'eps', 't']].to_numpy(dtype=np.float64)
    x_val = val_df[['u0', 'eps', 't']].to_numpy(dtype=np.float64)
    x_test = test_df[['u0', 'eps', 't']].to_numpy(dtype=np.float64)
    y_train_direct = train_df[['u_clean']].to_numpy(dtype=np.float64)
    y_val_direct = val_df[['u_clean']].to_numpy(dtype=np.float64)
    train_ou = u_ou_np(train_df['t'].to_numpy(), train_df['u0'].to_numpy())
    val_ou = u_ou_np(val_df['t'].to_numpy(), val_df['u0'].to_numpy())
    test_ou = u_ou_np(test_df['t'].to_numpy(), test_df['u0'].to_numpy())
    train_eps = train_df['eps'].to_numpy()
    val_eps = val_df['eps'].to_numpy()
    test_eps = test_df['eps'].to_numpy()
    y_train_res = ((train_df['u_clean'].to_numpy() - train_ou) / train_eps).reshape(-1, 1)
    y_val_res = ((val_df['u_clean'].to_numpy() - val_ou) / val_eps).reshape(-1, 1)

    for seed_idx in range(config['n_seeds']):
        run_seed = config['base_seed'] + seed_idx
        print(f'\nSeed {seed_idx:02d}')

        direct_model, direct_xs, direct_ys = train_scaled_model(
            x_train, y_train_direct, x_val, y_val_direct,
            seed=run_seed, epochs=config['epochs'], batch_size=config['batch_size'],
            lr=config['lr'], patience=config['patience'], device=device,
        )
        direct_pred = predict_scaled_model(direct_model, x_test, direct_xs, direct_ys, device=device).reshape(-1)
        pred_df = test_df.copy()
        pred_df['u_pred'] = direct_pred
        direct_metrics = compute_metrics(pred_df)
        all_rows.append({'model': 'Direct MLP', 'seed': seed_idx, **direct_metrics})

        residual_model, residual_xs, residual_ys = train_scaled_model(
            x_train, y_train_res, x_val, y_val_res,
            seed=run_seed + 5000, epochs=config['epochs'], batch_size=config['batch_size'],
            lr=config['lr'], patience=config['patience'], device=device,
        )
        residual_r_pred = predict_scaled_model(residual_model, x_test, residual_xs, residual_ys, device=device).reshape(-1)
        residual_pred = test_ou + test_eps * residual_r_pred
        pred_df = test_df.copy()
        pred_df['u_pred'] = residual_pred
        residual_metrics = compute_metrics(pred_df)
        all_rows.append({'model': 'Residual MLP', 'seed': seed_idx, **residual_metrics})

        print(f"  Direct E_traj={direct_metrics['E_traj']:.2e}, Residual E_traj={residual_metrics['E_traj']:.2e}")

        for lam in config['physics_lambdas']:
            pi_model, pi_xs, pi_rs = train_physics_informed_residual(
                train_df=train_df, val_df=val_df, lam=lam,
                seed=run_seed + int(lam * 10000) + 9000,
                epochs=config['epochs'], batch_size=config['batch_size'],
                lr=config['lr'], patience=config['patience'], device=device,
            )
            pi_pred = predict_physics_informed_residual(pi_model, x_test, pi_xs, pi_rs, device=device).reshape(-1)
            pred_df = test_df.copy()
            pred_df['u_pred'] = pi_pred
            pi_metrics = compute_metrics(pred_df)
            model_name = f'Physics-informed Residual MLP, lambda={lam}'
            all_rows.append({'model': model_name, 'seed': seed_idx, **pi_metrics})
            print(f"  PI lambda={lam}: E_traj={pi_metrics['E_traj']:.2e}")
    return pd.DataFrame(all_rows)


physics_seed_results = run_physics_baseline_comparison()
physics_seed_results.head()



Seed 00
  Direct E_traj=1.04e-07, Residual E_traj=4.05e-09
  PI lambda=0.01: E_traj=3.64e-09
  PI lambda=0.1: E_traj=1.33e-08
  PI lambda=1.0: E_traj=6.23e-08

Seed 01
  Direct E_traj=1.04e-07, Residual E_traj=3.24e-09
  PI lambda=0.01: E_traj=4.32e-09
  PI lambda=0.1: E_traj=1.84e-08
  PI lambda=1.0: E_traj=7.82e-08

Seed 02
  Direct E_traj=1.06e-07, Residual E_traj=5.60e-09
  PI lambda=0.01: E_traj=4.62e-09
  PI lambda=0.1: E_traj=2.67e-08
  PI lambda=1.0: E_traj=9.67e-08

Seed 03
  Direct E_traj=1.43e-07, Residual E_traj=7.05e-09
  PI lambda=0.01: E_traj=4.08e-09
  PI lambda=0.1: E_traj=1.62e-08
  PI lambda=1.0: E_traj=4.99e-08

Seed 04
  Direct E_traj=1.16e-07, Residual E_traj=5.08e-09
  PI lambda=0.01: E_traj=3.19e-09
  PI lambda=0.1: E_traj=9.25e-09
  PI lambda=1.0: E_traj=4.01e-08

Seed 05
  Direct E_traj=2.28e-07, Residual E_traj=4.02e-09
  PI lambda=0.01: E_traj=1.17e-08
  PI lambda=0.1: E_traj=2.76e-08
  PI lambda=1.0: E_traj=1.24e-07

Seed 06
  Direct E_traj=1.38e-07, Resid

,model,seed,E_traj,E_cross,E_over,E_eq
0,Direct MLP,0,1.044728e-07,0.001061,0.000348,0.000300
1,Residual MLP,0,4.051397e-09,0.000218,0.000064,0.000029
2,"Physics-informed Residual MLP, lambda=0.01",0,3.635182e-09,0.000207,0.000078,0.000043
3,"Physics-informed Residual MLP, lambda=0.1",0,1.333785e-08,0.000381,0.000082,0.000085
4,"Physics-informed Residual MLP, lambda=1.0",0,6.228299e-08,0.001012,0.000295,0.000081


## Summarize Across Seeds

In [ ]:

def summarize_results(seed_df, n_seeds):
    if SCIPY_AVAILABLE:
        tcrit = float(student_t.ppf(0.975, n_seeds - 1))
    else:
        tcrit = 2.262
    rows = []
    model_order = [
        'Direct MLP',
        'Residual MLP',
        'Physics-informed Residual MLP, lambda=0.01',
        'Physics-informed Residual MLP, lambda=0.1',
        'Physics-informed Residual MLP, lambda=1.0',
    ]
    for model in model_order:
        g = seed_df[seed_df['model'] == model]
        if len(g) == 0:
            continue
        row = {'model': model}
        for metric in ['E_traj', 'E_cross', 'E_over', 'E_eq']:
            vals = g[metric].to_numpy(dtype=float)
            mean = np.nanmean(vals)
            std = np.nanstd(vals, ddof=1)
            n = np.sum(~np.isnan(vals))
            ci = tcrit * std / math.sqrt(n) if n > 1 else np.nan
            row[f'{metric}_mean'] = mean
            row[f'{metric}_ci'] = ci
        rows.append(row)
    return pd.DataFrame(rows)

physics_summary = summarize_results(physics_seed_results, CONFIG['n_seeds'])
physics_summary


,model,E_traj_mean,E_traj_ci,E_cross_mean,E_cross_ci,E_over_mean,E_over_ci,E_eq_mean,E_eq_ci
0,Direct MLP,1.273412e-07,2.992754e-08,0.001990,0.000412,0.000495,0.000119,0.000260,0.000044
1,Residual MLP,4.658921e-09,8.766622e-10,0.000241,0.000046,0.000053,0.000011,0.000051,0.000015
2,"Physics-informed Residual MLP, lambda=0.01",4.964545e-09,1.746347e-09,0.000339,0.000123,0.000059,0.000013,0.000049,0.000008
3,"Physics-informed Residual MLP, lambda=0.1",2.021646e-08,6.370761e-09,0.000649,0.000156,0.000108,0.000034,0.000068,0.000019
4,"Physics-informed Residual MLP, lambda=1.0",1.303910e-07,1.011564e-07,0.001332,0.000293,0.000396,0.000230,0.000096,0.000027


## Generate LaTeX Table

In [ ]:
def fmt_metric(mean, ci):
    if np.isnan(mean):
        return "N/A"
    if np.isnan(ci):
        return f"${mean:.3e}$"
    return f"${mean:.3e} \\pm {ci:.1e}$"


def latex_model_name(model):
    if model == "Physics-informed Residual MLP, lambda=0.01":
        return r"Physics-informed Residual MLP, $\lambda=0.01$"
    if model == "Physics-informed Residual MLP, lambda=0.1":
        return r"Physics-informed Residual MLP, $\lambda=0.1$"
    if model == "Physics-informed Residual MLP, lambda=1.0":
        return r"Physics-informed Residual MLP, $\lambda=1.0$"
    return model


def make_physics_latex_table(summary_df):
    lines = []
    lines.append(r"\begin{table}[t]")
    lines.append(r"\centering")
    lines.append(r"\scriptsize")
    lines.append(
        r"\caption{Comparison with physics-informed residual MLP baselines. The physics-informed models add an ODE residual penalty to the trajectory loss using $\lambda\in\{0.01,0.1,1.0\}$. Results are reported as mean $\pm$ 95\% confidence-interval half-width over $S=10$ random seeds. Lower values indicate better performance.}"
    )
    lines.append(r"\label{tab:physics_informed_residual}")
    lines.append(r"\resizebox{\textwidth}{!}{%")
    lines.append(r"\begin{tabular}{lcccc}")
    lines.append(r"\hline")
    lines.append(
        r"Model & $E_{\mathrm{traj}}$ & $E_{\mathrm{cross}}$ & $E_{\mathrm{over}}$ & $E_{\mathrm{eq}}$ \\"
    )
    lines.append(r"\hline")

    for _, row in summary_df.iterrows():
        vals = [
            fmt_metric(row["E_traj_mean"], row["E_traj_ci"]),
            fmt_metric(row["E_cross_mean"], row["E_cross_ci"]),
            fmt_metric(row["E_over_mean"], row["E_over_ci"]),
            fmt_metric(row["E_eq_mean"], row["E_eq_ci"]),
        ]

        lines.append(
            f"{latex_model_name(row['model'])} & {vals[0]} & {vals[1]} & {vals[2]} & {vals[3]} \\\\"
        )

    lines.append(r"\hline")
    lines.append(r"\end{tabular}%")
    lines.append(r"}")
    lines.append(r"\end{table}")

    return "\n".join(lines)


physics_latex_table = make_physics_latex_table(physics_summary)
print(physics_latex_table)

\begin{table}[t]
\centering
\scriptsize
\caption{Comparison with physics-informed residual MLP baselines. The physics-informed models add an ODE residual penalty to the trajectory loss using $\lambda\in\{0.01,0.1,1.0\}$. Results are reported as mean $\pm$ 95\% confidence-interval half-width over $S=10$ random seeds. Lower values indicate better performance.}
\label{tab:physics_informed_residual}
\resizebox{\textwidth}{!}{%
\begin{tabular}{lcccc}
\hline
Model & $E_{\mathrm{traj}}$ & $E_{\mathrm{cross}}$ & $E_{\mathrm{over}}$ & $E_{\mathrm{eq}}$ \\
\hline
Direct MLP & $1.273e-07 \pm 3.0e-08$ & $1.990e-03 \pm 4.1e-04$ & $4.952e-04 \pm 1.2e-04$ & $2.602e-04 \pm 4.4e-05$ \\
Residual MLP & $4.659e-09 \pm 8.8e-10$ & $2.411e-04 \pm 4.6e-05$ & $5.256e-05 \pm 1.1e-05$ & $5.149e-05 \pm 1.5e-05$ \\
Physics-informed Residual MLP, $\lambda=0.01$ & $4.965e-09 \pm 1.7e-09$ & $3.389e-04 \pm 1.2e-04$ & $5.928e-05 \pm 1.3e-05$ & $4.919e-05 \pm 8.3e-06$ \\
Physics-informed Residual MLP, $\lambda=0.1$ & $2

## Save Results

In [ ]:

physics_seed_results.to_csv('physics_informed_residual_seed_results.csv', index=False)
physics_summary.to_csv('physics_informed_residual_summary.csv', index=False)
with open('physics_informed_residual_latex_table.tex', 'w', encoding='utf-8') as f:
    f.write(physics_latex_table)
print('Saved:')
print('  physics_informed_residual_seed_results.csv')
print('  physics_informed_residual_summary.csv')
print('  physics_informed_residual_latex_table.tex')


Saved:
  physics_informed_residual_seed_results.csv
  physics_informed_residual_summary.csv
  physics_informed_residual_latex_table.tex
